# Logistic Regression Baseline: event_adverse (1,275-day landmark)
Target: adverse event (died or removed too sick) within 1,275 days of listing
Data: `task3_encoded_unscaled.csv` from encoding-scaling-cells.ipynb; numeric features scaled after the split
Cohorts:
- **Main (corrected):** keeps patients with a known non-adverse exit (transplanted, transplanted elsewhere, removed administratively); drops only patients still waiting or `unknown` before 1,275 days
- **Sensitivity (original):** also drops patients with a known non-adverse exit before 1,275 days

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

In [ ]:
LANDMARK_DAYS = 1275
CORRECTED = True  # True = main cohort, False = sensitivity cohort

In [ ]:
df = pd.read_csv("../data/processed/task3_encoded_scaled.csv")
df.shape

In [ ]:
df = df[df["days_to_event"] > 0]

mask = (df["days_to_event"] >= LANDMARK_DAYS) | (df["event_adverse"] == 1)
if CORRECTED:
    mask |= df["outcome"].isin(["transplanted", "transplanted_elsewhere", "removed_administrative"])
df = df[mask]

y = ((df["event_adverse"] == 1) & (df["days_to_event"] <= LANDMARK_DAYS)).astype(int)

print("Main (corrected)" if CORRECTED else "Sensitivity (original)")
print(df.shape)
print(f"Adverse rate: {y.mean():.3f}")

In [ ]:
target_cols = ["outcome", "event_adverse", "event_transplant", "censored", "days_to_event"]
X = df.drop(columns=target_cols + ["DIALYSIS_DURATION_LOG"])
X.columns.tolist()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
X_train, X_test = X_train.copy(), X_test.copy()

In [ ]:
num_cols = ["BMI_TCR", "INIT_CPRA", "INIT_AGE", "functional_percent",
            "DIALYSIS_DURATION", "AGE_DIALYSIS_INTERACTION"]

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

In [ ]:
probs = model.predict_proba(X_test)[:, 1]
rate = y_test.mean()

print(f"Adverse rate: {rate:.3f}")
print(f"ROC-AUC: {roc_auc_score(y_test, probs):.3f}")
print(f"PR-AUC:  {average_precision_score(y_test, probs):.3f}")
print(f"Brier:   {brier_score_loss(y_test, probs):.3f}")

In [ ]:
results = pd.DataFrame({"actual": y_test, "prob": probs})
top10 = results.nlargest(len(results) // 10, "prob")
print(f"Top-decile lift: {top10['actual'].mean() / rate:.2f}x")

In [ ]:
coefs = pd.Series(model.coef_[0], index=X.columns).sort_values()
coefs

In [ ]:
odds_ratios = np.exp(coefs).sort_values()
odds_ratios